# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [82]:
!pip install 'tensorflow>2.0'

In [83]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline

# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы.

In [84]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

def load_mnist(num_val=0.4, num_test=0.2):
  digits = load_digits()
  X = digits.images
  y = digits.target
  classes = digits.target_names

  print('Images shape:', X.shape)
  print('Labels shape:', y.shape)
  print('Classes:', classes)

  # X_flat = X.reshape(X.shape[0], -1).astype(np.float64)

  X_train_full, X_test, y_train_full, y_test = train_test_split(
      X, y, test_size=num_test, random_state=42, stratify=y
  )
  X_train, X_val, y_train, y_val = train_test_split(
      X_train_full, y_train_full, test_size=num_val, random_state=42, stratify=y_train_full
  )

  #нормализация значений
  mean_image = np.mean(X_train, axis=0)
  X_train -= mean_image
  X_test -= mean_image
  X_val -= mean_image

  X_train = X_train[..., np.newaxis]
  X_val = X_val[..., np.newaxis]
  X_test = X_test[..., np.newaxis]


  print('Training data shape: ', X_train.shape)
  print('Training labels shape: ', y_train.shape)
  print('Validation data shape: ', X_val.shape)
  print('Validation labels shape: ', y_val.shape)
  print('Test data shape: ', X_test.shape)
  print('Test labels shape: ', y_test.shape)
  return X_train, y_train, X_val, y_val, X_test, y_test

In [85]:
X_train, y_train, X_val, y_val, X_test, y_test = load_mnist()

Images shape: (1797, 8, 8)
Labels shape: (1797,)
Classes: [0 1 2 3 4 5 6 7 8 9]
Training data shape:  (862, 8, 8, 1)
Training labels shape:  (862,)
Validation data shape:  (575, 8, 8, 1)
Validation labels shape:  (575,)
Test data shape:  (360, 8, 8, 1)
Test labels shape:  (360,)


In [86]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y

        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[idxs[i:i+B]], self.y[idxs[i:i+B]]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [87]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 8, 8, 1) (64,)
1 (64, 8, 8, 1) (64,)
2 (64, 8, 8, 1) (64,)
3 (64, 8, 8, 1) (64,)
4 (64, 8, 8, 1) (64,)
5 (64, 8, 8, 1) (64,)
6 (64, 8, 8, 1) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети.

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [88]:
device = '/GPU:0'

In [89]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации.

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU
5. Полносвязный слой
6. Функция активации Softmax

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [90]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****)
        self.conv1 = tf.keras.layers.Conv2D(filters=channel_1, kernel_size=(5,5), padding='same', activation='relu')
        self.conv2 = tf.keras.layers.Conv2D(filters=channel_2, kernel_size=(3,3), padding='same', activation='relu')
        self.flatten = tf.keras.layers.Flatten()
        self.dense = tf.keras.layers.Dense(units=num_classes, activation='softmax')

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################

    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        if len(x.shape) == 3:
                x = tf.expand_dims(x, axis=-1)

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.dense(x)
        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        return scores

In [91]:
def test_ThreeLayerConvNet():
    x = tf.zeros((64, 8, 8))
    model = ThreeLayerConvNet(channel_1=32, channel_2=16, num_classes=10)
    scores = model(x)
    print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [92]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.

    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for

    Returns: Nothing, but prints progress during trainingn
    """
    with tf.device(device):


        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

        model = model_init_fn()
        optimizer = optimizer_init_fn()

        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')

        t = 0
        for epoch in range(num_epochs):

            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()

            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:

                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)

                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)

                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)

                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [93]:
hidden_size, num_classes = 100, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

print_every = 10
train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 8.903778076171875, Accuracy: 18.75, Val Loss: 7.590052127838135, Val Accuracy: 18.434782028198242
Iteration 10, Epoch 1, Loss: 4.25313138961792, Accuracy: 36.505680084228516, Val Loss: 1.6692967414855957, Val Accuracy: 60.173912048339844


Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 .

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [94]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1=channel_1, channel_2=channel_2, num_classes=num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
      learning_rate=learning_rate, momentum=0.9, nesterov=True, name='SGD')

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.429399251937866, Accuracy: 17.1875, Val Loss: 2.440253734588623, Val Accuracy: 15.30434799194336
Iteration 10, Epoch 1, Loss: 1.8910564184188843, Accuracy: 42.755680084228516, Val Loss: 1.1258920431137085, Val Accuracy: 75.47826385498047


# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [95]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (8, 8, 1)
    hidden_layer_size, num_classes = 400, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax',
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 11.167000770568848, Accuracy: 10.9375, Val Loss: 6.637234210968018, Val Accuracy: 19.130434036254883
Iteration 10, Epoch 1, Loss: 3.252568244934082, Accuracy: 53.97727584838867, Val Loss: 1.1145316362380981, Val Accuracy: 74.95652770996094


Альтернативный менее гибкий способ обучения:

In [96]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 96ms/step - loss: 3.3614 - sparse_categorical_accuracy: 0.4942 - val_loss: 1.0709 - val_sparse_categorical_accuracy: 0.7791
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 1.1177 - sparse_categorical_accuracy: 0.7694


[1.11765456199646, 0.769444465637207]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [97]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    input_shape = (8, 8, 1)
    channel_1, channel_2, dense = 32, 16, 10
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv2D(channel_1, (5, 5), padding='same', activation='relu'),
        tf.keras.layers.Conv2D(channel_2, (3, 3), padding='same', activation='relu'),
        # преобразование в одномерный вектор перед полносвязным слоем
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(dense, activation='softmax')
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 5e-4
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    optimizer = tf.keras.optimizers.SGD(
      learning_rate=learning_rate, momentum=0.9, nesterov=True, name='SGD')

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.5133633613586426, Accuracy: 1.5625, Val Loss: 2.457484245300293, Val Accuracy: 8.17391300201416
Iteration 10, Epoch 1, Loss: 2.32271409034729, Accuracy: 12.5, Val Loss: 2.114487886428833, Val Accuracy: 26.782608032226562


In [98]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

14/14 ━━━━━━━━━━━━━━━━━━━━ 7s 270ms/step - loss: 1.7080 - sparse_categorical_accuracy: 0.4745 - val_loss: 1.0387 - val_sparse_categorical_accuracy: 0.8070
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step - loss: 1.0346 - sparse_categorical_accuracy: 0.7917


[1.0345666408538818, 0.7916666865348816]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры.

Ниже представлен пример для полносвязной сети.

In [99]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)

    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)

    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_two_layer_fc_functional()

(64, 10)


In [100]:
input_shape = (8, 8, 1)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 9.660951614379883, Accuracy: 1.5625, Val Loss: 11.759705543518066, Val Accuracy: 47.4782600402832
Iteration 10, Epoch 1, Loss: 7.394408702850342, Accuracy: 66.05113983154297, Val Loss: 1.1328123807907104, Val Accuracy: 89.7391357421875


Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут).

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [101]:
class CNNExperiment:
    def __init__(self, name):
        self.name = name
        self.history = None
        self.model = None

    def build_model(self, architecture_config):
        """Динамическое построение модели на основе конфигурации"""
        model = tf.keras.Sequential(name=self.name)
        model.add(tf.keras.layers.Input(shape=(8, 8, 1)))

        # сверточные слои
        for i, conv in enumerate(architecture_config.get('conv_layers', [])):
            model.add(tf.keras.layers.Conv2D(
                filters=conv['filters'],
                kernel_size=conv.get('kernel_size', 3),
                activation=conv.get('activation', 'relu'),
                padding=conv.get('padding', 'same')
            ))

            if conv.get('batch_norm', False):
                model.add(tf.keras.layers.BatchNormalization())

            if conv.get('pooling', True):
                model.add(tf.keras.layers.MaxPooling2D(2))

            if conv.get('dropout', 0):
                model.add(tf.keras.layers.Dropout(conv['dropout']))

        # полносвязные слои
        model.add(tf.keras.layers.Flatten())

        for dense in architecture_config.get('dense_layers', []):
            model.add(tf.keras.layers.Dense(
                units=dense['units'],
                activation=dense.get('activation', 'relu')
            ))
            if dense.get('dropout', 0):
                model.add(tf.keras.layers.Dropout(dense['dropout']))

        model.add(tf.keras.layers.Dense(
            architecture_config.get('num_classes', 10),
            activation='softmax'
        ))

        self.model = model
        return model

    def compile_model(self, model, compile_config):
        """Компиляция модели с разными параметрами"""
        optimizer = self._get_optimizer(compile_config)

        model.compile(
            optimizer=optimizer,
            loss=compile_config.get('loss', 'sparse_categorical_crossentropy'),
            metrics=['accuracy']
        )
        return model

    def _get_optimizer(self, config):
        """Выбор оптимизатора"""
        opt_name = config.get('optimizer', 'adam').lower()
        lr = config.get('learning_rate', 0.001)

        if opt_name == 'adam':
            return tf.keras.optimizers.Adam(learning_rate=lr)
        elif opt_name == 'sgd':
            momentum = config.get('momentum', 0.9)
            return tf.keras.optimizers.SGD(learning_rate=lr, momentum=momentum)
        else:
            return tf.keras.optimizers.Adam(learning_rate=lr)

    def train(self, x_train, y_train, x_val, y_val, train_config):
        """Обучение модели"""
        callbacks = []
        history = self.model.fit(
            x_train, y_train,
            epochs=train_config.get('epochs', 10),
            batch_size=train_config.get('batch_size', 32),
            validation_data=(x_val, y_val),
            callbacks=callbacks,
            verbose=1
        )

        self.history = history
        return history

In [102]:
experiments = {
    # Простая CNN
    'simple_cnn': {
        'architecture': {
            'conv_layers': [
                {'filters': 32, 'kernel_size': 3, 'pooling': True, 'dropout': 0}
            ],
            'dense_layers': [
                {'units': 128, 'dropout': 0.5}
            ]
        },
        'compile': {
            'optimizer': 'adam',
            'learning_rate': 0.001,
            'loss': 'sparse_categorical_crossentropy'
        },
        'train': {
            'epochs': 10,
            'batch_size': 32,
            'early_stopping': False
        }
    },

    # Более глубокая CNN
    'deep_cnn': {
        'architecture': {
            'conv_layers': [
                {'filters': 32, 'kernel_size': 3, 'pooling': True},
                {'filters': 64, 'kernel_size': 3, 'pooling': True},
                {'filters': 64, 'kernel_size': 3, 'pooling': False, 'dropout': 0.25}
            ],
            'dense_layers': [
                {'units': 256, 'dropout': 0.5},
                {'units': 128, 'dropout': 0.3}
            ]
        },
        'compile': {
            'optimizer': 'adam',
            'learning_rate': 0.001
        },
        'train': {
            'epochs': 10,
            'batch_size': 32
        }
    },

    # CNN с BatchNorm
    'cnn_with_batchnorm': {
        'architecture': {
            'conv_layers': [
                {'filters': 32, 'kernel_size': 3, 'batch_norm': True, 'pooling': True},
                {'filters': 64, 'kernel_size': 3, 'batch_norm': True, 'pooling': True},
                {'filters': 128, 'kernel_size': 3, 'batch_norm': True, 'pooling': False}
            ],
            'dense_layers': [
                {'units': 256, 'dropout': 0.5},
                {'units': 128, 'dropout': 0.3}
            ]
        },
        'compile': {
            'optimizer': 'adam',
            'learning_rate': 0.001
        },
        'train': {
            'epochs': 10,
            'batch_size': 32
        }
    },

    # CNN с SGD и моментом
    'cnn_with_sgd': {
        'architecture': {
            'conv_layers': [
                {'filters': 64, 'kernel_size': 3, 'pooling': True, 'dropout': 0.25},
                {'filters': 128, 'kernel_size': 3, 'pooling': True, 'dropout': 0.25}
            ],
            'dense_layers': [
                {'units': 512, 'dropout': 0.5}
            ]
        },
        'compile': {
            'optimizer': 'sgd',
            'learning_rate': 0.01,
            'momentum': 0.9
        },
        'train': {
            'epochs': 10,
            'batch_size': 32
        }
    }
}

In [103]:
def run_experiments(experiments_dict, x_train, y_train, x_val, y_val):
    results = {}

    for exp_name, config in experiments_dict.items():
        exp = CNNExperiment(exp_name)

        model = exp.build_model(config['architecture'])
        print(f"Архитектура: {model.summary()}")

        model = exp.compile_model(model, config['compile'])

        history = exp.train(x_train, y_train, x_val, y_val, config['train'])

        results[exp_name] = {
            'history': history.history,
            'best_val_acc': max(history.history['val_accuracy']),
            'best_val_loss': min(history.history['val_loss']),
            'model': exp.model,
            'config': config
        }

        final_acc = history.history['val_accuracy'][-1]
        print(f"Точность: {final_acc:.2%}")
    print()
    return results

In [104]:
results = run_experiments(experiments, X_train, y_train, X_test, y_test)

Model: "simple_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_130 (Conv2D)             │ (None, 8, 8, 32)       │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_73 (MaxPooling2D) │ (None, 4, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_62 (Flatten)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_141 (Dense)               │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_83 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_142 (Dense)               │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67,274 (262.79 KB)

 Trainable params: 67,274 (262.79 KB)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_137 (Conv2D)             │ (None, 8, 8, 64)       │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_78 (MaxPooling2D) │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_89 (Dropout)            │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_138 (Conv2D)             │ (None, 4, 4, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_79 (MaxPooling2D) │ (None, 2, 2, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_90 (Dropout)            │ (None, 2, 2, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_65 (Flatten)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_149 (Dense)               │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_91 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_150 (Dense)               │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 342,282 (1.31 MB)

 Trainable params: 342,282 (1.31 MB)

 Non-trainable params: 0 (0.00 B)

Архитектура: None
Epoch 1/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 7s 157ms/step - accuracy: 0.3213 - loss: 1.9884 - val_accuracy: 0.8028 - val_loss: 0.7129
Epoch 2/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7425 - loss: 0.7615 - val_accuracy: 0.8667 - val_loss: 0.3476
Epoch 3/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8527 - loss: 0.4493 - val_accuracy: 0.9333 - val_loss: 0.2203
Epoch 4/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8759 - loss: 0.3517 - val_accuracy: 0.9500 - val_loss: 0.1803
Epoch 5/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9188 - loss: 0.2551 - val_accuracy: 0.9750 - val_loss: 0.1000
Epoch 6/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9281 - loss: 0.2008 - val_accuracy: 0.9722 - val_loss: 0.0911
Epoch 7/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9455 - loss: 0.1523 - val_accuracy: 0.9583 - val_loss: 0.1053
Epoch 8/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9478 - loss: 0.1218 - val_accuracy

In [105]:
best_exp_name = max(results.keys(), key=lambda x: results[x]['best_val_acc'])
best_accuracy = results[best_exp_name]['best_val_acc']

print(f"Лучший эксперимент: {best_exp_name}")
print(f"Лучшая точность: {best_accuracy:.4f}")

Лучший эксперимент: simple_cnn
Лучшая точность: 0.9778


Максимальное значение точности для набора simple_cnn достигает значения 0.975.

Теперь протестируем на наборе без функций активации.

In [106]:
experiments_withot_activation_func = {
    # Простая CNN
    'simple_cnn': {
        'architecture': {
            'conv_layers': [
                {'filters': 32, 'kernel_size': 3, 'pooling': True, 'dropout': 0}
            ],
            'dense_layers': [
                {'units': 10, 'dropout': 0.5}
            ]
        },
        'compile': {
            'optimizer': 'adam',
            'learning_rate': 0.001,
            'loss': 'sparse_categorical_crossentropy'
        },
        'train': {
            'epochs': 10,
            'batch_size': 32,
            'early_stopping': False
        }
    },

    # Более глубокая CNN
    'deep_cnn': {
        'architecture': {
            'conv_layers': [
                {'filters': 32, 'kernel_size': 3, 'pooling': True},
                {'filters': 64, 'kernel_size': 3, 'pooling': True},
                {'filters': 64, 'kernel_size': 3, 'pooling': False, 'dropout': 0.25}
            ],
            'dense_layers': [
                {'units': 256, 'dropout': 0.5},
                {'units': 10, 'dropout': 0.3}
            ]
        },
        'compile': {
            'optimizer': 'adam',
            'learning_rate': 0.001
        },
        'train': {
            'epochs': 10,
            'batch_size': 32
        }
    },

    # CNN с BatchNorm
    'cnn_with_batchnorm': {
        'architecture': {
            'conv_layers': [
                {'filters': 32, 'kernel_size': 3, 'batch_norm': True, 'pooling': True},
                {'filters': 64, 'kernel_size': 3, 'batch_norm': True, 'pooling': True},
                {'filters': 128, 'kernel_size': 3, 'batch_norm': True, 'pooling': False}
            ],
            'dense_layers': [
                {'units': 256, 'dropout': 0.5},
                {'units': 10, 'dropout': 0.3}
            ]
        },
        'compile': {
            'optimizer': 'adam',
            'learning_rate': 0.001
        },
        'train': {
            'epochs': 10,
            'batch_size': 32
        }
    },

    # CNN с SGD и моментом
    'cnn_with_sgd': {
        'architecture': {
            'conv_layers': [
                {'filters': 64, 'kernel_size': 3, 'pooling': True, 'dropout': 0.25},
                {'filters': 128, 'kernel_size': 3, 'pooling': True, 'dropout': 0.25}
            ],
            'dense_layers': [
                {'units': 10, 'dropout': 0.5}
            ]
        },
        'compile': {
            'optimizer': 'sgd',
            'learning_rate': 0.01,
            'momentum': 0.9
        },
        'train': {
            'epochs': 10,
            'batch_size': 32
        }
    }
}

In [107]:
results_without_activation_func = run_experiments(experiments_withot_activation_func, X_train, y_train, X_val, y_val)

Model: "simple_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_139 (Conv2D)             │ (None, 8, 8, 32)       │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_80 (MaxPooling2D) │ (None, 4, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_66 (Flatten)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_151 (Dense)               │ (None, 10)             │         5,130 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_92 (Dropout)            │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_152 (Dense)               │ (None, 10)             │           110 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,560 (21.72 KB)

 Trainable params: 5,560 (21.72 KB)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_146 (Conv2D)             │ (None, 8, 8, 64)       │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_85 (MaxPooling2D) │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_98 (Dropout)            │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_147 (Conv2D)             │ (None, 4, 4, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_86 (MaxPooling2D) │ (None, 2, 2, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_99 (Dropout)            │ (None, 2, 2, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_69 (Flatten)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_159 (Dense)               │ (None, 10)             │         5,130 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_100 (Dropout)           │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_160 (Dense)               │ (None, 10)             │           110 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,736 (311.47 KB)

 Trainable params: 79,736 (311.47 KB)

 Non-trainable params: 0 (0.00 B)

Архитектура: None
Epoch 1/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 9s 207ms/step - accuracy: 0.1032 - loss: 2.3519 - val_accuracy: 0.1009 - val_loss: 2.2454
Epoch 2/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0928 - loss: 2.2868 - val_accuracy: 0.1009 - val_loss: 2.3025
Epoch 3/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0974 - loss: 2.2833 - val_accuracy: 0.1791 - val_loss: 2.1302
Epoch 4/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1392 - loss: 2.1779 - val_accuracy: 0.1948 - val_loss: 2.0000
Epoch 5/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1555 - loss: 2.1533 - val_accuracy: 0.2817 - val_loss: 1.9467
Epoch 6/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1694 - loss: 2.0588 - val_accuracy: 0.2661 - val_loss: 1.8240
Epoch 7/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1961 - loss: 2.0299 - val_accuracy: 0.2800 - val_loss: 1.8801
Epoch 8/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.1868 - loss: 2.0195 - val_accurac

In [108]:
best_exp_without_activation_func_name = max(results_without_activation_func.keys(), key=lambda x: results_without_activation_func[x]['best_val_acc'])
best_without_activation_func_accuracy = results_without_activation_func[best_exp_without_activation_func_name]['best_val_acc']

print(f"Лучший эксперимент: {best_exp_without_activation_func_name}")
print(f"Лучшая точность: {best_without_activation_func_accuracy:.4f}")

Лучший эксперимент: deep_cnn
Лучшая точность: 0.9670


In [111]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        self.conv1 = tf.keras.layers.Conv2D(32, kernel_size=3, use_bias=False,
            padding='same', kernel_initializer=initializer)
        self.bn1 = tf.keras.layers.BatchNormalization()

        self.conv2 = tf.keras.layers.Conv2D(64, kernel_size=3, use_bias=False,
            padding='same', kernel_initializer=initializer)
        self.bn2 = tf.keras.layers.BatchNormalization()

        self.drop1 = tf.keras.layers.Dropout(0.25)

        self.conv3 = tf.keras.layers.Conv2D(128, kernel_size=3, use_bias=False,
            padding='same', kernel_initializer=initializer)
        self.bn3 = tf.keras.layers.BatchNormalization()

        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(128, activation='relu', kernel_initializer=initializer)
        self.drop2 = tf.keras.layers.Dropout(0.5)
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax')

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################

    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(input_tensor)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)

        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)

        x = self.drop1(x, training=training)

        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = tf.nn.relu(x)

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.drop2(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


num_epochs = 10
print_every = 10

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate)

trained_model = train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 3.1573472023010254, Accuracy: 6.25, Val Loss: 6.670790672302246, Val Accuracy: 31.478261947631836
Iteration 10, Epoch 1, Loss: 1.5754562616348267, Accuracy: 52.6988639831543, Val Loss: 0.7886901497840881, Val Accuracy: 83.13043975830078
Iteration 20, Epoch 2, Loss: 0.47966617345809937, Accuracy: 84.59821319580078, Val Loss: 0.6053903102874756, Val Accuracy: 86.9565200805664
Iteration 30, Epoch 3, Loss: 0.1690053939819336, Accuracy: 95.83332824707031, Val Loss: 0.542974054813385, Val Accuracy: 88.69564819335938
Iteration 40, Epoch 3, Loss: 0.21167929470539093, Accuracy: 93.38941955566406, Val Loss: 0.26298072934150696, Val Accuracy: 93.0434799194336
Iteration 50, Epoch 4, Loss: 0.17104020714759827, Accuracy: 94.09722137451172, Val Loss: 0.2905031740665436, Val Accuracy: 92.86956787109375
Iteration 60, Epoch 5, Loss: 0.11861857026815414, Accuracy: 95.0, Val Loss: 0.21245907247066498, Val Accuracy: 93.73912811279297
Iteration 70, Epoch 6, Loss: 0.08847659826278

Опишите все эксперименты, результаты. Сделайте выводы.

Для решения задачи была разработана нейронная сеть CustomConvNet. Ее архитектура состоит из трех сверточных слоев, за которыми следуют слои пакетной нормализации (BatchNormalization) и регуляризации методом Dropout. Завершают сеть два полносвязных слоя. Обучение модели осуществлялось с помощью оптимизатора Adam с фиксированной скоростью обучения 0.001. Пакетная нормализация применялась для стабилизации и ускорения процесса обучения путем нормализации активаций в каждом батче, а Dropout служил для предотвращения переобучения путем случайного отключения нейронов. Тренировка модели заняла 10 эпох. Уже ко второй эпохе точность на обучающей выборке составила 84,6%, а к концу обучения достигла 97,5% на валидационной и тестовой выборках.